# Vectorised thinking in xarray

A working notebook on four ideas that come up constantly in ensemble climate analysis:

1. **Pointwise indexing**: using `.sel` / `.isel` with a DataArray indexer as a lookup, instead of a `groupby` or a loop
2. **Vector thinking**: weighted sums, `xr.dot`, and spotting when an operation is a linear operator you can build once and apply everywhere
3. **`apply_ufunc` and `vectorize=True`**: what a "true" ufunc is, why `gaussian_kde` isn't one, and what that costs you
4. **`xr.map_blocks`**: what it hands your function, when it's the right tool, and how it fails silently

Each section has a short explanation and runnable examples. Section 5 has 12 questions. **The answers are at the very bottom, behind a spoiler wall.** Try the questions in the empty cells first.

Everything runs on synthetic data, top to bottom, in well under a minute.

In [ ]:
import time

import numpy as np
import pandas as pd
import xarray as xr
from scipy.stats import gaussian_kde, linregress
from statsmodels.nonparametric.smoothers_lowess import lowess

## 0. Synthetic data

Three arrays, shaped like things you actually work with:

* `daily_tas`: 30 years of daily data on a small Antarctic grid, with a seasonal cycle that peaks in January
* `annual_tas`: 152 years of annual means (1856–2007) with a warming trend, four volcanic dips and noise
* `ens_annual`: a 20-member annual ensemble on a 4 × 5 grid

In [ ]:
rng = np.random.default_rng(42)

daily_time = pd.date_range("1990-01-01", "2019-12-31", freq="D")
lat = np.linspace(-80, -62, 10)
lon = np.arange(0, 360, 30.0)
seasonal_cycle = 10 * np.cos(2 * np.pi * (daily_time.dayofyear.values - 15) / 365.25)
daily_tas = xr.DataArray(
    seasonal_cycle[:, None, None] + rng.normal(0, 3, (daily_time.size, lat.size, lon.size)),
    dims=("time", "lat", "lon"),
    coords={"time": daily_time, "lat": lat, "lon": lon},
    name="tas",
)

years = np.arange(1856, 2008)
annual_lat = np.linspace(-89, -51, 20)
annual_lon = np.arange(0, 360, 10.0)
forced = 0.01 * (years - years[0]) - 0.5 * np.isin(years, [1883, 1963, 1982, 1991])
annual_tas = xr.DataArray(
    forced[:, None, None] + rng.normal(0, 0.4, (years.size, annual_lat.size, annual_lon.size)),
    dims=("year", "lat", "lon"),
    coords={"year": years, "lat": annual_lat, "lon": annual_lon},
    name="tas",
)

ens_annual = xr.DataArray(
    forced[None, :, None, None] + rng.normal(0, 0.4, (20, years.size, 4, 5)),
    dims=("member", "year", "lat", "lon"),
    coords={
        "member": [f"r{i}i1p1f3" for i in range(1, 21)],
        "year": years,
        "lat": annual_lat[:4],
        "lon": annual_lon[:5],
    },
    name="tas",
)

print("daily_tas :", dict(daily_tas.sizes))
print("annual_tas:", dict(annual_tas.sizes))
print("ens_annual:", dict(ens_annual.sizes))

## 1. Pointwise indexing: a lookup instead of a groupby

There are three different ways two xarray objects can interact, and mixing them up is the source of most "why is this array suddenly 900 million elements" moments.

**Broadcasting (arithmetic and comparisons).** Dims are matched *by name*. Any dim that exists in only one of the two arrays is combined with the other as an outer product. So `daily_tas > season_q`, where one has `time` and the other has `season`, doesn't match days to seasons. It compares every day against every season.

**Orthogonal indexing (`.sel` with lists or NumPy arrays).** Each dim is selected independently. `da.sel(lat=[a, b, c], lon=[x, y, z])` gives you a 3 × 3 block of 9 points, not 3 points.

**Pointwise, or vectorised, indexing (`.sel` / `.isel` with a DataArray indexer).** The indexer carries its own dims, and the result swaps the indexed dim for the indexer's dims. It's a *gather*: for every position in the indexer, go and fetch the matching label.

The rule that makes it predictable:

> `a.sel(dim=indexer)` has the dims of `a`, with `dim` replaced by `indexer.dims`. The indexer's coords come along for the ride.

That's why `q.sel(season=da.time.dt.season)` works. `da.time.dt.season` is a DataArray with dim `time` and one season label per day. Selecting with it replaces `season` with `time`, so the threshold array lines up with the data day for day. One vectorised operation, no loop, no groupby.

The "extending a dimension" trick is the same rule used deliberately: pass an indexer with *new* dim names (say `boot` and `draw`), and the indexed dim is replaced by those new dims.

### Example 1.1: seasonal thresholds, the wrong way and the right way

In [ ]:
season_q = daily_tas.groupby("time.season").quantile([0.9, 0.99], dim="time")
print("season_q            :", dict(season_q.sizes))

broadcast_wrong = daily_tas > season_q
print("daily_tas > season_q:", dict(broadcast_wrong.sizes), f"= {broadcast_wrong.size:,} booleans")

Every day has been compared with all four seasons' thresholds. Now the gather:

In [ ]:
season_thresh = season_q.sel(season=daily_tas.time.dt.season)
print("gathered threshold:", dict(season_thresh.sizes))
season_thresh.coords

`season` has gone from a dimension to a non-index coordinate along `time`, one label per day. It stays attached through the comparison, so you can still group by it afterwards. Exceedance frequency should come out close to 0.10 and 0.01:

In [ ]:
season_exceed = daily_tas > season_thresh
season_exceed.groupby("season").mean(["time", "lat", "lon"])

### Example 1.2: day-of-year anomalies, gather vs groupby

Both give the same answer, and eagerly they take about the same time. The gather version is an ordinary array operation on the original `time` axis, which is easier to reason about and to reuse. Where the difference really shows is on Dask (Example 4.1).

In [ ]:
doy_clim = daily_tas.groupby("time.dayofyear").mean("time")

t0 = time.perf_counter()
doy_anom_gather = daily_tas - doy_clim.sel(dayofyear=daily_tas.time.dt.dayofyear)
t_gather = time.perf_counter() - t0

t0 = time.perf_counter()
doy_anom_groupby = daily_tas.groupby("time.dayofyear") - doy_clim
t_groupby = time.perf_counter() - t0

xr.testing.assert_allclose(doy_anom_gather, doy_anom_groupby)
print(f"identical results | gather {t_gather * 1e3:.1f} ms | groupby {t_groupby * 1e3:.1f} ms")

### Example 1.3: station extraction, orthogonal vs pointwise

Five stations, each with its own lat and lon.

In [ ]:
stn_lat = [-78, -75, -70, -66, -63]
stn_lon = [30, 100, 170, 250, 320]

stations_ortho = daily_tas.sel(lat=stn_lat, lon=stn_lon, method="nearest")
print("orthogonal (lists):", dict(stations_ortho.sizes))

stations = xr.Dataset(
    {"lat": ("station", stn_lat), "lon": ("station", stn_lon)},
    coords={"station": ["A", "B", "C", "D", "E"]},
)
stations_point = daily_tas.sel(lat=stations.lat, lon=stations.lon, method="nearest")
print("pointwise (DataArray indexers):", dict(stations_point.sizes))
stations_point.coords

The orthogonal version gives 25 series, most of them at places you didn't ask for. The pointwise version gives 5, and each carries the grid `lat` and `lon` it actually snapped to as coordinates along `station`.

### Example 1.4: extending a dimension on purpose (bootstrap without a loop)

A 2-D integer indexer with dims `(boot, draw)` replaces `member` with both of them.

In [ ]:
draw_idx = xr.DataArray(
    rng.integers(0, ens_annual.sizes["member"], (500, 10)),
    dims=("boot", "draw"),
)
boot_sample = ens_annual.isel(member=draw_idx)
print("ens_annual :", dict(ens_annual.sizes))
print("boot_sample:", dict(boot_sample.sizes))
print("member coord is now", boot_sample.member.dims, "so you can see which members were drawn")

boot_mean_spread = boot_sample.mean("draw").std("boot")
print("typical spread of a 10-member mean:", round(float(boot_mean_spread.mean()), 3))

Two things to keep in mind. The gathered array is `boot × draw / member` times the size of the original, 250× here, so it grows fast. And on Dask, a gather along `member` pulls whole chunks, so chunk with `member=-1` first or you'll get a lot of shuffling.

## 2. Vector thinking: weighted sums, `xr.dot`, and linear operators

A loop over gridcells asks "what do I do to this one series?". Vector thinking asks "what operation, applied to the whole array at once, gives me every answer?".

A surprising number of operations are **weighted sums along one dimension**: a mean, an area average, a trend slope, the difference between two periods, a running mean, a projection onto an EOF. A weighted sum over a dim is a dot product, $\sum_i w_i y_i$.

`xr.dot(a, b, dim=...)` multiplies and sums over the named dims and broadcasts everything else. It's `np.einsum` with names instead of letters. (Older xarray versions spell the argument `dims=`.)

**Linear operators.** Go one step further. If every output value is a weighted sum of the inputs, with weights that depend only on the x-axis (years, positions) and not on the data, then the whole operation is a matrix:

$$\hat{\mathbf{y}} = \mathbf{S}\,\mathbf{y}$$

Build **S** once, which is cheap because it only depends on the axis, and apply it to every gridcell with one matrix multiply that runs in optimised BLAS. That's the trick behind the fast LOWESS: local linear regression with tricube weights (`it=0`) has an **S**, so you can skip fitting 720 separate regressions.

**The linearity test.** If `f(a*y1 + b*y2) == a*f(y1) + b*f(y2)` for any series, then an **S** exists. If the weights depend on the data itself (robustness reweighting, medians, dividing by the series' own standard deviation) there is no single **S**.

### Example 2.1: area-weighted mean as a dot product

In [ ]:
area_w = np.cos(np.deg2rad(annual_tas.lat))
area_w_2d = area_w.broadcast_like(annual_tas.isel(year=0, drop=True))
area_w_norm = area_w_2d / area_w_2d.sum()

area_mean_dot = xr.dot(annual_tas, area_w_norm, dim=["lat", "lon"])
area_mean_weighted = annual_tas.weighted(area_w).mean(["lat", "lon"])

print("dims:", area_mean_dot.dims)
print("max difference:", float(abs(area_mean_dot - area_mean_weighted).max()))

### Example 2.2: the OLS slope is a dot product

The least-squares slope is

$$b = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}{\sum_i (x_i - \bar{x})^2}$$

Because $\sum_i (x_i - \bar{x}) = 0$, the $\bar{y}$ term drops out, and the slope is just $\sum_i c_i y_i$ with $c_i = (x_i - \bar{x}) / \sum_j (x_j - \bar{x})^2$. The weights $c_i$ only depend on the years, so one weight vector gives you the trend at every gridcell.

In [ ]:
year_x = annual_tas.year.astype(float)
year_anom = year_x - year_x.mean()
slope_w = year_anom / (year_anom**2).sum()

t0 = time.perf_counter()
slope_dot = xr.dot(annual_tas, slope_w, dim="year")
t_dot = time.perf_counter() - t0

t0 = time.perf_counter()
slope_polyfit = annual_tas.polyfit("year", deg=1).polyfit_coefficients.sel(degree=1, drop=True)
t_polyfit = time.perf_counter() - t0

print("max difference:", float(abs(slope_dot - slope_polyfit).max()))
print(f"dot {t_dot * 1e3:.1f} ms | polyfit {t_polyfit * 1e3:.1f} ms")

### Example 2.3: a running mean is a matrix

Row *i* of **S** holds the weights that produce smoothed value *i*. For a centred running mean, that's 1/*k* for the *k* points in the window and 0 everywhere else. Here's a tiny one, 8 points with a 3-point window, truncated at the ends:

In [ ]:
def running_mean_matrix(n, window):
    """Build the matrix form of a centred running mean.

    Args:
        n (int): Length of the series.
        window (int): Odd window width; windows are truncated at the ends.

    Returns:
        np.ndarray: (n, n) matrix whose row i averages the points within the window around i.
    """
    distance = np.abs(np.arange(n)[:, None] - np.arange(n)[None, :])
    inside = (distance <= window // 2).astype(float)
    return inside / inside.sum(axis=1, keepdims=True)


np.round(running_mean_matrix(8, 3), 2)

To apply it with `xr.dot`, **S** needs two dims: the output `year` and the input `year_src`. Rename the data's `year` to `year_src` so the dot product sums over the right one.

In [ ]:
n_years = annual_tas.sizes["year"]
year_pair_coords = {"year": annual_tas.year.values, "year_src": annual_tas.year.values}

runmean_da = xr.DataArray(
    running_mean_matrix(n_years, 11), dims=("year", "year_src"), coords=year_pair_coords
)
annual_src = annual_tas.rename(year="year_src")

runmean_dot = xr.dot(runmean_da, annual_src, dim="year_src").transpose("year", "lat", "lon")
runmean_rolling = annual_tas.rolling(year=11, center=True, min_periods=1).mean()
print("max difference vs rolling:", float(abs(runmean_dot - runmean_rolling).max()))

### Example 2.4: LOWESS (`it=0`) as a hat matrix

For each target year *i*, LOWESS fits a weighted straight line to the *r* nearest points and reads off the fitted value at *i*. The weights are tricube in distance. Written as algebra, the fitted value is the first row of $(\mathbf{X}^\top \mathbf{W}\mathbf{X})^{-1}\mathbf{X}^\top \mathbf{W}$ dotted with the window's *y* values. Nothing in that row depends on *y*, so each row of **S** can be computed once.

The loop below runs over the 152 years of the *axis*, once. The expensive loop, over gridcells, is gone.

In [ ]:
def lowess_hat_matrix(n, window):
    """Build the smoother matrix for LOWESS with no robustness iterations on an even grid.

    Args:
        n (int): Length of the series.
        window (int): Number of points in each local regression.

    Returns:
        np.ndarray: (n, n) matrix S such that S @ y is the LOWESS fit of y.
    """
    x = np.arange(n, dtype=float)
    start = np.clip(np.arange(n) - window // 2, 0, n - window)
    hat = np.zeros((n, n))
    for i in range(n):
        idx = np.arange(start[i], start[i] + window)
        dist = x[idx] - x[i]
        weights = (1 - np.abs(dist / np.abs(dist).max()) ** 3) ** 3
        design = np.column_stack([np.ones(window), dist])
        weighted_design = design * weights[:, None]
        hat[i, idx] = np.linalg.solve(design.T @ weighted_design, weighted_design.T)[0]
    return hat


def lowess_1d(y, frac):
    """Smooth one series with statsmodels LOWESS and no robustness iterations.

    Args:
        y (np.ndarray): 1-D series on an evenly spaced axis.
        frac (float): Fraction of points in each local window.

    Returns:
        np.ndarray: Smoothed series in the original order.
    """
    return lowess(y, np.arange(y.size, dtype=float), frac=frac, it=0, delta=0, return_sorted=False)

In [ ]:
lowess_window = 31
hat_da = xr.DataArray(
    lowess_hat_matrix(n_years, lowess_window), dims=("year", "year_src"), coords=year_pair_coords
)

t0 = time.perf_counter()
lowess_dot = xr.dot(hat_da, annual_src, dim="year_src").transpose("year", "lat", "lon")
t_dot = time.perf_counter() - t0

t0 = time.perf_counter()
lowess_loop = xr.apply_ufunc(
    lowess_1d,
    annual_tas,
    kwargs={"frac": lowess_window / n_years},
    input_core_dims=[["year"]],
    output_core_dims=[["year"]],
    vectorize=True,
).transpose("year", "lat", "lon")
t_loop = time.perf_counter() - t0

print("max difference vs statsmodels:", float(abs(lowess_dot - lowess_loop).max()))
print(f"hat matrix {t_dot * 1e3:.1f} ms | statsmodels per gridcell {t_loop * 1e3:.0f} ms")

### Example 2.5: the linearity test in practice

`it=0` passes. `it=3` (the statsmodels default, with bisquare robustness reweighting) fails, because the robustness weights are computed from the residuals, which depend on *y*. That's why the matrix trick only works for `it=0`.

In [ ]:
test_y1 = annual_tas.isel(lat=0, lon=0).values
test_y2 = annual_tas.isel(lat=5, lon=9).values
test_x = np.arange(n_years, dtype=float)

for n_iter in [0, 3]:
    smooth_sum = lowess(test_y1 + test_y2, test_x, frac=0.2, it=n_iter, delta=0, return_sorted=False)
    sum_smooth = (
        lowess(test_y1, test_x, frac=0.2, it=n_iter, delta=0, return_sorted=False)
        + lowess(test_y2, test_x, frac=0.2, it=n_iter, delta=0, return_sorted=False)
    )
    print(f"it={n_iter}: max |f(y1 + y2) - (f(y1) + f(y2))| = {np.abs(smooth_sum - sum_smooth).max():.2e}")

One caveat to carry forward: a matrix multiply has opinions about NaNs. Question 8 is about exactly that.

## 3. `apply_ufunc` and what `vectorize=True` really does

**What a "true" ufunc is.** A NumPy universal function (`np.exp`, `np.add`, `np.maximum`) is compiled code that works element by element on arrays of *any* shape. The looping happens in C. A *generalised* ufunc (gufunc) extends that idea to core dimensions: `np.matmul` has the signature `(n,k),(k,m)->(n,m)` and loops over any leading dims in C as well.

**What `apply_ufunc` assumes.** It's named after that model. It moves each input's `input_core_dims` to the *end* of the array and hands your function plain NumPy arrays of shape `(..., core)`. It assumes your function deals with the leading `...` itself, the way `np.mean(arr, axis=-1)` does.

**Why `gaussian_kde` isn't one.** `scipy.stats.gaussian_kde(dataset)` wants one sample. Pass it a 2-D array and it doesn't loop over rows. It assumes each row is a separate *variable* in a multivariate KDE. Same story for `statsmodels` `lowess` (1-D `x` and `y`) and `scipy.stats.linregress` (1-D). They don't have a gufunc signature, so they can't handle leading dims.

**What `vectorize=True` does about it.** It wraps your function in `np.vectorize`, which the NumPy docs describe as essentially a for loop. Every gridcell becomes a separate Python-level call. `apply_ufunc` makes that *convenient*, and `dask="parallelized"` spreads chunks across workers, but inside each chunk it's still a Python loop.

**The fast route.** If you can rewrite the function to work along the last axis for any leading shape, you can drop `vectorize=True` and let NumPy broadcast. For a KDE that's a sum of Gaussian kernels, which broadcasts naturally.

### Example 3.1: what your function actually receives

In [ ]:
def show_shape(arr):
    """Print the shape apply_ufunc passes in, then average over the last axis.

    Args:
        arr (np.ndarray): Array with the core dim last.

    Returns:
        np.ndarray: Mean over the last axis.
    """
    print("function received shape", arr.shape)
    return arr.mean(axis=-1)


print("daily_tas dims:", daily_tas.dims, daily_tas.shape)
time_mean_ufunc = xr.apply_ufunc(show_shape, daily_tas, input_core_dims=[["time"]])
print("result dims   :", time_mean_ufunc.dims)

`time` was moved to the end, and the function got the whole array in one call. Because it used `axis=-1`, it handled the leading `(lat, lon)` itself. That's the gufunc-like case: `vectorize=True` isn't needed.

### Example 3.2: counting the calls with `vectorize=True`

In [ ]:
call_counter = {"n": 0}


def count_calls(arr):
    """Count invocations and return the mean of a 1-D array.

    Args:
        arr (np.ndarray): 1-D series.

    Returns:
        float: Mean of the series.
    """
    call_counter["n"] += 1
    return arr.mean()


xr.apply_ufunc(count_calls, daily_tas, input_core_dims=[["time"]], vectorize=True)
print("calls:", call_counter["n"], "| gridcells:", daily_tas.sizes["lat"] * daily_tas.sizes["lon"])

One Python call per gridcell. At 120 cells nobody notices. At 20 lats × 144 lons × 7 quantiles × 4 seasons it's over 80,000 calls.

### Example 3.3: `gaussian_kde` with a 2-D input

It doesn't error when you build it. It errors (or worse, quietly gives the wrong thing) when you evaluate it:

In [ ]:
kde_samples = xr.DataArray(
    rng.gamma(4.0, 1.5, (lat.size, lon.size, 400)),
    dims=("lat", "lon", "sample"),
    coords={"lat": lat, "lon": lon},
)
kde_x = np.linspace(0, 25, 200)

kde_2d_wrong = gaussian_kde(kde_samples.values.reshape(-1, 400))
print(f"gaussian_kde thinks it has {kde_2d_wrong.d} variables and {kde_2d_wrong.n} points")
try:
    kde_2d_wrong(kde_x)
except ValueError as err:
    print("ValueError:", err)

### Example 3.4: the loop version vs the broadcast version

`kde_nd` computes the same thing as `gaussian_kde` (Gaussian kernels, Scott's bandwidth) but along the last axis for any leading shape. The intermediate array is `(..., n_sample, n_x)`, so you pay in memory what you save in Python calls.

Whether that trade is worth it depends on how much work each call does, so the comparison below runs two regimes.

In [ ]:
def kde_1d(sample, grid):
    """Evaluate scipy's Gaussian KDE of one sample on a grid.

    Args:
        sample (np.ndarray): 1-D sample.
        grid (np.ndarray): Points at which to evaluate the density.

    Returns:
        np.ndarray: Density at each grid point.
    """
    return gaussian_kde(sample)(grid)


def kde_nd(sample, grid):
    """Evaluate a Gaussian KDE along the last axis with Scott's bandwidth, for any leading shape.

    Args:
        sample (np.ndarray): Array of shape (..., n) holding one sample per leading index.
        grid (np.ndarray): 1-D array of m points at which to evaluate the density.

    Returns:
        np.ndarray: Array of shape (..., m) holding the density for each leading index.
    """
    n = sample.shape[-1]
    bandwidth = sample.std(axis=-1, ddof=1, keepdims=True) * n ** (-1 / 5)
    z = (grid - sample[..., None]) / bandwidth[..., None]
    return np.exp(-0.5 * z**2).sum(axis=-2) / (n * bandwidth * np.sqrt(2 * np.pi))

In [ ]:
kde_kwargs = {
    "kwargs": {"grid": kde_x},
    "input_core_dims": [["sample"]],
    "output_core_dims": [["x"]],
}
kde_regimes = {
    "120 cells, 400-value samples": kde_samples,
    "720 cells, 30-value samples": xr.DataArray(rng.gamma(4.0, 1.5, (20, 36, 30)), dims=("lat", "lon", "sample")),
}

for label, regime_samples in kde_regimes.items():
    xr.apply_ufunc(kde_nd, regime_samples, **kde_kwargs)

    t0 = time.perf_counter()
    regime_loop = xr.apply_ufunc(kde_1d, regime_samples, vectorize=True, **kde_kwargs)
    t_loop = time.perf_counter() - t0

    t0 = time.perf_counter()
    regime_broadcast = xr.apply_ufunc(kde_nd, regime_samples, **kde_kwargs)
    t_broadcast = time.perf_counter() - t0

    max_diff = float(abs(regime_loop - regime_broadcast).max())
    print(f"{label:30s} loop {t_loop * 1e3:5.0f} ms | broadcast {t_broadcast * 1e3:5.0f} ms | max diff {max_diff:.1e}")

Same answer to floating point in both cases, but very different payoffs.

* **120 cells, 400-value samples.** Each `gaussian_kde` call already does 400 × 200 of vectorised work internally, so the Python overhead of the loop is a small share of the cost. The broadcast version gains little or nothing, and builds a big temporary array to do it.
* **720 cells, 30-value samples** (think 30 annual maxima per gridcell). Each call does very little work, so the per-call overhead dominates, and broadcasting wins clearly (around 3× here).

It's the same lesson as the robust-LOWESS experiment, where a fully vectorised version was only about 12% faster than a statsmodels loop: vectorising removes *per-call overhead*, so it pays in proportion to how much of the runtime that overhead was. Time it before rewriting.

### Example 3.5: the same thing on Dask

With `dask="parallelized"` and a *new* output dim (`x` didn't exist on the input), Dask needs to be told its size up front via `output_sizes`.

In [ ]:
kde_lazy = xr.apply_ufunc(
    kde_nd,
    kde_samples.chunk({"lat": 5}),
    dask="parallelized",
    output_dtypes=[float],
    dask_gufunc_kwargs={"output_sizes": {"x": kde_x.size}},
    **kde_kwargs,
)
kde_eager = xr.apply_ufunc(kde_1d, kde_samples, vectorize=True, **kde_kwargs)
print(kde_lazy.chunks)
print("matches scipy:", bool(np.allclose(kde_lazy.compute(), kde_eager)))

**Summary for section 3.**

* Function already works along `axis=-1` for any leading shape: `vectorize=False` (the default). Fast.
* Function only takes 1-D input and can be rewritten with broadcasting: rewrite it, then `vectorize=False`. Usually a big win when each call is cheap, small or none when each call is already heavy (Example 3.4).
* Function genuinely can't be vectorised (an optimiser per series, like a GEV fit): `vectorize=True` is correct, and `dask="parallelized"` is how you claw back speed, by running chunks in parallel. The loop is the price of the function, not a mistake.
* `numba.guvectorize` is the other escape hatch: it compiles a 1-D kernel into a real gufunc, so the loop runs in machine code.

## 4. `xr.map_blocks`: when and why

**What it hands your function.** `apply_ufunc(..., dask="parallelized")` gives your function NumPy blocks. `xr.map_blocks(func, obj)` gives your function each Dask block as a full xarray object, with coords, names and attrs. So inside `func` you can use `.sel`, `groupby`, `polyfit`, anything xarray.

**The contract.** Each call sees *one block only*. There's no overlap with neighbours. So anything that needs to see a whole dimension (a climatology over time, a quantile over time, a rolling window along a chunked dim) needs that dimension in a single chunk. If it isn't, `map_blocks` won't complain. It'll give you a wrong answer.

**The template.** xarray has to know what the output looks like before running anything. Without `template=` it tries running `func` on a zero-size mock object, which fails for most real functions (`quantile`, `polyfit`). So in practice you pass a template: a (lazy) object with the output's dims, coords and chunks. It's cheap to build from the input: `da.isel(time=0, drop=True)` for a reduction over time, or `.expand_dims(...)` to add a new dim.

**When to reach for it.**

* Built-in xarray methods are already Dask-aware. Use those first.
* `apply_ufunc` is the right tool when your function works on NumPy arrays.
* `map_blocks` is the right tool when your function is written in xarray terms, each block is self-contained, and you want to run it block by block. Typical cases: wrapping an existing xarray function that isn't Dask-friendly, or a multi-step pipeline where you want one task per block instead of a long chain of small tasks.

### Example 4.1: per-block day-of-year anomaly (time in one chunk)

In [ ]:
def doy_anomaly(block):
    """Subtract the block's own day-of-year climatology.

    Args:
        block (xr.DataArray): Daily data with a complete time axis.

    Returns:
        xr.DataArray: Anomalies with the same shape and coords as the input.
    """
    clim = block.groupby("time.dayofyear").mean("time")
    return (block - clim.sel(dayofyear=block.time.dt.dayofyear)).drop_vars("dayofyear")


daily_space_chunked = daily_tas.chunk({"time": -1, "lat": 5, "lon": 6})
doy_anom_blocks = xr.map_blocks(doy_anomaly, daily_space_chunked, template=daily_space_chunked)

doy_anom_chained = daily_space_chunked - daily_space_chunked.groupby("time.dayofyear").mean("time").sel(
    dayofyear=daily_space_chunked.time.dt.dayofyear
)

print("tasks, map_blocks:", len(doy_anom_blocks.__dask_graph__()))
print("tasks, chained   :", len(doy_anom_chained.__dask_graph__()))
print("matches eager:", bool(np.allclose(doy_anom_blocks.compute(), doy_anom_gather)))

Two things to notice.

The `.drop_vars("dayofyear")` is required. The gather leaves `dayofyear` behind as a coordinate along `time` (as in Example 1.1), and `map_blocks` checks every block's output against the template exactly. A stray coordinate that isn't on the template raises `ValueError: Result from applying user function has unexpected coordinate variables`. That strictness is annoying the first time and helpful after that.

The task counts. The chained version builds a long graph of small Dask operations for the groupby and the gather. `map_blocks` runs the whole function as one task per block. A graph that size costs real time to build and schedule before any arithmetic happens.

### Example 4.2: the silent failure (time chunked)

Same function, but the data is chunked along `time` in 5-year blocks. Each block computes its climatology from its own 5 years. No error, no warning.

In [ ]:
daily_time_chunked = daily_tas.chunk({"time": 365 * 5})
doy_anom_wrong = xr.map_blocks(doy_anomaly, daily_time_chunked, template=daily_time_chunked)

print("time chunks:", daily_time_chunked.chunks[0])
print("max difference from the correct anomaly:", round(float(abs(doy_anom_wrong - doy_anom_gather).max()), 2))

### Example 4.3: a reduction, with a template

`polyfit` inside each block, one trend per gridcell. The output has no `year` dim, so the template is the input with `year` dropped.

In [ ]:
def linear_trend(block):
    """Fit a straight line along year and return the slope.

    Args:
        block (xr.DataArray): Annual data with a year dim.

    Returns:
        xr.DataArray: Slope per unit year, with year removed.
    """
    return block.polyfit("year", deg=1).polyfit_coefficients.sel(degree=1, drop=True)


annual_chunked = annual_tas.chunk({"year": -1, "lat": 10, "lon": 12})
trend_template = annual_chunked.isel(year=0, drop=True)
trend_blocks = xr.map_blocks(linear_trend, annual_chunked, template=trend_template)

print("template:", dict(trend_template.sizes), trend_template.chunks)
print("matches the dot-product slope:", bool(np.allclose(trend_blocks.compute(), slope_dot)))

Of course, the dot-product slope from section 2 is simpler still, and fully Dask-aware without `map_blocks`. That's a pattern worth noticing: once something is a weighted sum, you rarely need any of the machinery.

## 5. Questions

Work in the empty cell under each question. Several ask you to predict something *before* running code, which is where most of the learning is. All the arrays from sections 0–4 are available. The cell directly below sets up a couple of extra objects the questions use.

In [ ]:
q_stations = xr.Dataset(
    {"lat": ("station", [-79.5, -79.6, -70.2, -64.0]), "lon": ("station", [2.0, 8.0, 185.0, 300.0])},
    coords={"station": ["Kohnen", "Neighbour", "Ross", "Peninsula"]},
)

annual_gappy = annual_tas.copy()
annual_gappy.loc[{"year": 1906, "lat": annual_lat[0], "lon": annual_lon[0]}] = np.nan

### Q1. Predict the dims

Without running it: what are the dims and sizes of `doy_clim.sel(dayofyear=daily_tas.time.dt.dayofyear)`? What happens to the `dayofyear` coordinate? Then check.

In [ ]:
# Q1

### Q2. The plausible-looking mistake

A colleague runs `(daily_tas > season_q).mean("time")` and gets numbers without any error.

(a) What are the dims of the result?
(b) In words, what does the `season="DJF", quantile=0.9` slice actually measure?
(c) Roughly what value would you expect for that slice, and for `season="JJA", quantile=0.9`? Then check.

In [ ]:
# Q2

### Q3. Stations

Using `q_stations`:

(a) How many series does `daily_tas.sel(lat=q_stations.lat.values, lon=q_stations.lon.values, method="nearest")` return, and why?
(b) Write the pointwise selection that returns one series per station, with dim `station`.
(c) "Kohnen" and "Neighbour" are close together. Do they end up in the same gridcell? What does pointwise selection do about duplicates?

In [ ]:
# Q3

### Q4. Bootstrap without replacement

From `ens_annual`, draw 1000 resamples of 10 members **without** replacement (10 distinct members per resample) and compute the spread (std over resamples) of the 10-member mean, per year and gridcell. No loops.

Also: how many times bigger than `ens_annual` is the gathered array before you take the mean?

In [ ]:
# Q4

### Q5. `xr.dot` and NaNs

`annual_gappy` has one NaN, at `year=1906` in the corner gridcell.

(a) Predict what `xr.dot(annual_gappy, area_w_norm, dim=["lat", "lon"])` gives for 1906, and what `annual_gappy.weighted(area_w).mean(["lat", "lon"])` gives.
(b) Write a dot-product version that skips NaNs the way `.weighted().mean()` does.

In [ ]:
# Q5

### Q6. A period difference as one weight vector

Write "mean of the last 30 years minus mean of the first 30 years" as a single weight vector over `year`, and apply it with `xr.dot` to `annual_tas`. Check it against the direct calculation.

In [ ]:
# Q6

### Q7. Is there an S?

For each operation, say whether it can be written as one matrix **S** applied to every gridcell's time series, and why:

(a) 11-year running mean
(b) 11-year running median
(c) LOWESS with `it=0`
(d) LOWESS with `it=3`
(e) removing a linear trend
(f) anomaly relative to the 1961–1990 mean
(g) standardising each series by its own standard deviation

In [ ]:
# Q7 (answer in words, or test with the linearity check from Example 2.5)

### Q8. The hat matrix and a single NaN

`hat_da` is the LOWESS matrix from Example 2.4 (window 31). `annual_gappy` has one NaN at 1906 in the corner gridcell.

Predict: after `xr.dot(hat_da, annual_gappy.rename(year="year_src"), dim="year_src")`, how many of the 152 smoothed years are NaN in that gridcell? Then check, explain it, and suggest a fix.

In [ ]:
# Q8

### Q9. A function that isn't a ufunc

```python
def slope_1d(y):
    return linregress(np.arange(y.size), y).slope
```

(a) Why does this need `vectorize=True` in `apply_ufunc`?
(b) Write `slope_nd(y)` that works along the last axis for any leading shape, use it with `vectorize=False` on `annual_tas`, and time both.

In [ ]:
# Q9

### Q10. Dimension order

`daily_tas` has dims `(time, lat, lon)`. For

```python
xr.apply_ufunc(f, daily_tas, input_core_dims=[["lat"]], output_core_dims=[["lat"]])
```

(a) What shape does `f` receive?
(b) What dims does the result have, and how do you get back to `(time, lat, lon)`?

In [ ]:
# Q10

### Q11. `map_blocks` with a new dim

Use `xr.map_blocks` on `daily_space_chunked` to compute the 0.9 and 0.99 quantiles over `time` for each gridcell. Build the template yourself, then check your answer against `daily_tas.quantile(...)`.

In [ ]:
# Q11

### Q12. Pick the tool

For each task, choose from: a built-in xarray method, pointwise indexing, `xr.dot`, `apply_ufunc` with `vectorize=False`, `apply_ufunc` with `vectorize=True`, or `xr.map_blocks`. Give a one-line reason.

(a) Subtract a monthly climatology from monthly data.
(b) Fit a GEV per gridcell with `scipy.stats.genextreme.fit`.
(c) A 21-year running mean on a `(time, lat, lon)` Dask array chunked along time.
(d) Pull ERA5 values at 300 AWS station locations.
(e) Run an existing xarray function (it uses `groupby` and `polyfit` internally) over a 50 GB array where every gridcell is independent.
(f) Project each year's field onto an EOF pattern.

In [ ]:
# Q12 (answer in words)

<br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br>

**Answers below. Scroll on only once you've had a go.**

<br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br><br>

## 6. Answers

### A1

`(time: 10957, lat: 10, lon: 12)`. The indexer `daily_tas.time.dt.dayofyear` has dim `time`, so `dayofyear` is replaced by `time`. The `dayofyear` coordinate survives as a non-index coordinate along `time`, one value per day, which you can drop with `.drop_vars("dayofyear")` if it gets in the way of later alignment.

In [ ]:
a1_gathered = doy_clim.sel(dayofyear=daily_tas.time.dt.dayofyear)
print(dict(a1_gathered.sizes))
print("dayofyear dims:", a1_gathered.dayofyear.dims)

### A2

(a) `(lat, lon, season, quantile)`. `time` was averaged out; `season` and `quantile` came from the outer-product broadcast.

(b) The fraction of **all days of the year**, not just DJF days, that are above the DJF 90th percentile.

(c) DJF is the warm season here, so few days outside summer beat its 90th percentile: expect a few percent. JJA is the cold season, so most of the year beats its 90th percentile: expect well over half. Neither is anywhere near 0.1, which is what you'd see if the comparison had been done right.

In [ ]:
a2_wrong = (daily_tas > season_q).mean("time")
print(a2_wrong.dims)
print("DJF 0.9:", round(float(a2_wrong.sel(season="DJF", quantile=0.9).mean()), 3))
print("JJA 0.9:", round(float(a2_wrong.sel(season="JJA", quantile=0.9).mean()), 3))

a2_right = (daily_tas > season_q.sel(season=daily_tas.time.dt.season)).groupby("season").mean(["time", "lat", "lon"])
print("correct:", a2_right.sel(quantile=0.9).round(3).values)

### A3

(a) 16 series. Plain NumPy arrays trigger orthogonal indexing: 4 lats × 4 lons, every combination.

(b) Pass the Dataset's variables, which are DataArrays with dim `station`.

(c) Yes, both snap to the same gridcell. Pointwise selection doesn't deduplicate: you get 4 series, two of them identical. That's usually what you want (one output per station), but worth knowing if you're averaging over stations afterwards.

In [ ]:
a3_ortho = daily_tas.sel(lat=q_stations.lat.values, lon=q_stations.lon.values, method="nearest")
print("(a)", dict(a3_ortho.sizes))

a3_point = daily_tas.sel(lat=q_stations.lat, lon=q_stations.lon, method="nearest")
print("(b)", dict(a3_point.sizes))

print("(c) snapped to:")
for name, snapped_lat, snapped_lon in zip(a3_point.station.values, a3_point.lat.values, a3_point.lon.values):
    print(f"    {name:10s} lat {snapped_lat:6.1f}  lon {snapped_lon:6.1f}")
print("Kohnen and Neighbour identical:", bool((a3_point.sel(station="Kohnen") == a3_point.sel(station="Neighbour")).all()))

### A4

`rng.integers` samples with replacement. For distinct members, rank a block of random numbers along each row and keep the first 10 positions: that gives a random permutation per resample.

Gathered size: 1000 × 10 / 20 = **500×** the original. The spread should sit around 0.09: noise sd 0.4, divided by √10, with a finite-population correction of √(10/19) because you're drawing half of a fixed ensemble.

In [ ]:
n_member = ens_annual.sizes["member"]
a4_idx = xr.DataArray(
    np.argsort(rng.random((1000, n_member)), axis=1)[:, :10],
    dims=("boot", "draw"),
)
a4_sample = ens_annual.isel(member=a4_idx)
print("size multiplier:", a4_sample.size / ens_annual.size)

a4_spread = a4_sample.mean("draw").std("boot")
print(dict(a4_spread.sizes))
print("mean spread:", round(float(a4_spread.mean()), 3), "| theory:", round(0.4 / np.sqrt(10) * np.sqrt(10 / 19), 3))
print("all draws distinct:", bool((np.sort(a4_idx.values, axis=1)[:, 1:] != np.sort(a4_idx.values, axis=1)[:, :-1]).all()))

### A5

(a) The dot product gives **NaN** for 1906: NaN × weight is NaN, and one NaN in a sum poisons it. `.weighted().mean()` **skips** the NaN and renormalises by the sum of the weights that are left.

(b) Fill NaNs with 0 in the numerator, and divide by the sum of weights over valid points only.

In [ ]:
a5_dot = xr.dot(annual_gappy, area_w_norm, dim=["lat", "lon"])
a5_weighted = annual_gappy.weighted(area_w).mean(["lat", "lon"])
print("(a) dot 1906:", float(a5_dot.sel(year=1906)), "| weighted 1906:", round(float(a5_weighted.sel(year=1906)), 4))

a5_valid = annual_gappy.notnull().astype(float)
a5_dot_skipna = xr.dot(annual_gappy.fillna(0), area_w_2d, dim=["lat", "lon"]) / xr.dot(a5_valid, area_w_2d, dim=["lat", "lon"])
print("(b) max difference vs weighted:", float(abs(a5_dot_skipna - a5_weighted).max()))

### A6

+1/30 on the last 30 years, −1/30 on the first 30, 0 in between. Same pattern as the slope: the weights only depend on the axis.

In [ ]:
a6_w = xr.zeros_like(annual_tas.year, dtype=float)
a6_w[-30:] = 1 / 30
a6_w[:30] = -1 / 30

a6_dot = xr.dot(annual_tas, a6_w, dim="year")
a6_direct = annual_tas.isel(year=slice(-30, None)).mean("year") - annual_tas.isel(year=slice(None, 30)).mean("year")
print("max difference:", float(abs(a6_dot - a6_direct).max()))

### A7

* (a) **Yes.** Fixed weights of 1/*k* inside the window.
* (b) **No.** Which point is the median depends on the values, so the "weights" change with *y*.
* (c) **Yes.** Tricube weights depend only on distances along the axis. That's the hat matrix in Example 2.4.
* (d) **No.** The bisquare robustness weights are computed from residuals, which depend on *y* (Example 2.5).
* (e) **Yes.** The fitted line is **H**y with **H** the OLS hat matrix, so the detrended series is (**I** − **H**)y.
* (f) **Yes.** Subtracting a fixed-period mean is (**I** − **1w**ᵀ)y, with **w** = 1/30 on 1961–1990.
* (g) **No.** You divide by a quantity computed from *y*, so doubling *y* doesn't double the output. It leaves it unchanged.

In [ ]:
a7_x = np.arange(n_years, dtype=float)
a7_y1 = annual_tas.isel(lat=0, lon=0).values
a7_y2 = annual_tas.isel(lat=5, lon=9).values
a7_ops = {
    "(b) running median": lambda y: pd.Series(y).rolling(11, center=True, min_periods=1).median().values,
    "(e) detrend": lambda y: y - np.polyval(np.polyfit(a7_x, y, 1), a7_x),
    "(g) standardise": lambda y: y / y.std(),
}
for name, op in a7_ops.items():
    gap = np.abs(op(a7_y1 + a7_y2) - (op(a7_y1) + op(a7_y2))).max()
    print(f"{name:20s} linearity gap {gap:.2e}")

### A8

**All 152.** Most people guess "the 31 years around 1906". But `hat_da` is a *dense* matrix: row *i* is mostly zeros, and 0 × NaN is still NaN. So the single NaN reaches every output, not just the ones whose window contains 1906. (With a *sparse* matrix, only the rows that actually store an entry for 1906 would go NaN, which is the "window width" behaviour.)

Fixes, depending on how honest you need it to be:

* Drop or mask gappy gridcells and handle them separately with statsmodels.
* Fill NaN with 0 and renormalise each row by the weight that fell on valid points: `S @ fillna(y, 0) / S @ notnull(y)`. Rows whose window doesn't touch the gap are then exactly unchanged. Rows that do are an approximation, not a true refit without that point, because the local-linear weights were solved with the missing point included.

In [ ]:
a8_src = annual_gappy.rename(year="year_src")
a8_smooth = xr.dot(hat_da, a8_src, dim="year_src")
print("NaN years in corner cell:", int(a8_smooth.isel(lat=0, lon=0).isnull().sum()))

a8_valid = a8_src.notnull().astype(float)
a8_renorm = xr.dot(hat_da, a8_src.fillna(0), dim="year_src") / xr.dot(hat_da, a8_valid, dim="year_src")
a8_changed = abs(a8_renorm - lowess_dot).isel(lat=0, lon=0) > 1e-12
print("NaN after renormalising:", int(a8_renorm.isel(lat=0, lon=0).isnull().sum()))
print("years that differ from the gap-free fit:", int(a8_changed.sum()), "between",
      int(a8_changed.year[a8_changed].min()), "and", int(a8_changed.year[a8_changed].max()))

### A9

(a) `linregress` expects 1-D `x` and `y`. It has no notion of "leading dims", so `apply_ufunc` has to call it once per gridcell, which is what `vectorize=True` arranges.

(b) Reuse the dot-product slope from Example 2.2, written along the last axis.

In [ ]:
def slope_1d(y):
    """OLS slope of one series against its index, via scipy.

    Args:
        y (np.ndarray): 1-D series.

    Returns:
        float: Slope per step.
    """
    return linregress(np.arange(y.size), y).slope


def slope_nd(y):
    """OLS slope along the last axis against the index, for any leading shape.

    Args:
        y (np.ndarray): Array of shape (..., n).

    Returns:
        np.ndarray: Slope per step, shape (...).
    """
    x_anom = np.arange(y.shape[-1]) - (y.shape[-1] - 1) / 2
    return y @ (x_anom / (x_anom**2).sum())


t0 = time.perf_counter()
a9_loop = xr.apply_ufunc(slope_1d, annual_tas, input_core_dims=[["year"]], vectorize=True)
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
a9_fast = xr.apply_ufunc(slope_nd, annual_tas, input_core_dims=[["year"]])
t_fast = time.perf_counter() - t0

print("max difference:", float(abs(a9_loop - a9_fast).max()))
print(f"vectorize=True {t_loop * 1e3:.0f} ms | slope_nd {t_fast * 1e3:.2f} ms")

### A10

(a) `(10957, 12, 10)`: `(time, lon, lat)`. Core dims go to the end, the rest keep their order.

(b) The result is `(time, lon, lat)`. Output core dims are placed last too. Use `.transpose(*daily_tas.dims)` to restore the original order. (That's the same thing that happened with `year` in the LOWESS example, and why those calls end in `.transpose(...)`.)

In [ ]:
def a10_shape(arr):
    """Print the incoming shape and return the array unchanged.

    Args:
        arr (np.ndarray): Array with the core dim last.

    Returns:
        np.ndarray: The input array.
    """
    print("received", arr.shape)
    return arr


a10_out = xr.apply_ufunc(a10_shape, daily_tas, input_core_dims=[["lat"]], output_core_dims=[["lat"]])
print("result dims:", a10_out.dims, "->", a10_out.transpose(*daily_tas.dims).dims)

### A11

The output drops `time` and gains `quantile` (size 2), which comes out as the leading dim from `.quantile`. Build the template by dropping `time` and expanding a `quantile` dim with the right coordinate values. `expand_dims` on a Dask array gives the new dim a single chunk, which is what `map_blocks` expects. `time` is in one chunk already, so each block sees the full record.

In [ ]:
a11_levels = [0.9, 0.99]


def time_quantiles(block):
    """Compute quantiles over time for one block.

    Args:
        block (xr.DataArray): Daily data with a complete time axis.

    Returns:
        xr.DataArray: Quantiles with dims (quantile, lat, lon).
    """
    return block.quantile(a11_levels, dim="time")


a11_template = daily_space_chunked.isel(time=0, drop=True).expand_dims(quantile=a11_levels)
a11_blocks = xr.map_blocks(time_quantiles, daily_space_chunked, template=a11_template)
print("template:", dict(a11_template.sizes), a11_template.chunks)
print("matches eager:", bool(np.allclose(a11_blocks.compute(), daily_tas.quantile(a11_levels, dim="time"))))

### A12

* (a) **Built-in or pointwise indexing.** `da.groupby("time.month") - clim` or `da - clim.sel(month=da.time.dt.month)`. Both are one line and Dask-aware.
* (b) **`apply_ufunc` with `vectorize=True`** plus `dask="parallelized"`. Each fit is an optimiser run with no closed form, so a per-gridcell loop is unavoidable. The speed comes from running chunks in parallel, not from vectorising.
* (c) **Built-in `.rolling()`.** Dask handles the overlap between time chunks for you. `map_blocks` would be wrong at every chunk boundary, the same way Example 4.2 was wrong.
* (d) **Pointwise indexing** with DataArray indexers along a `station` dim. Lists would give you a 300 × 300 orthogonal grid.
* (e) **`xr.map_blocks`**, with `time` in a single chunk and chunking in space. The function is written in xarray terms and each block is self-contained.
* (f) **`xr.dot`** over `lat` and `lon` (with area weights folded in). A projection is a weighted sum over space.